In [1]:
from google.colab import userdata
import os
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

In [2]:
!kaggle datasets download -d zeesolver/consumer-behavior-and-shopping-habits-dataset

Dataset URL: https://www.kaggle.com/datasets/zeesolver/consumer-behavior-and-shopping-habits-dataset
License(s): CC0-1.0
consumer-behavior-and-shopping-habits-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
! unzip "consumer-behavior-and-shopping-habits-dataset.zip"

Archive:  consumer-behavior-and-shopping-habits-dataset.zip
replace shopping_behavior_updated.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: shopping_behavior_updated.csv  
replace shopping_trends.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: shopping_trends.csv     


In [4]:
import pandas as pd

In [5]:
df=pd.read_csv("shopping_behavior_updated.csv")

In [6]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


# First Prompt

Create a graph that presents which age (in bins) and gender group spend more money and on what. Exactly the same only with Plotly

In [7]:
# prompt: Using dataframe df: Create a graph that presents which age (in bins) and gender group spend more money and on what. Exactly the same only with Plotly

import pandas as pd
import plotly.express as px

# Create bins for age
df['Age Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65, 70],
                        labels=['18-24', '25-34', '35-44', '45-54', '55-64', '65-70'])

# Group data by age group, gender, and category, and sum purchase amount
grouped_df = df.groupby(['Age Group', 'Gender', 'Category'])['Purchase Amount (USD)'].sum().reset_index()

# Create the Plotly bar chart
fig = px.bar(grouped_df, x='Age Group', y='Purchase Amount (USD)', color='Category',
             facet_col='Gender', barmode='group',
             title='Spending by Age Group, Gender, and Category')

fig.show()


<ipython-input-7-2e3cfb97d370>:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_df = df.groupby(['Age Group', 'Gender', 'Category'])['Purchase Amount (USD)'].sum().reset_index()


# Second Prompt

Analyze my df and find the best-selling items (item purchased) for each category. You will need to count sum of sails for each item. Create a pie chart for each category.

In [8]:
# prompt: Analyze my df and find the best-selling items (item purchased) for each category. You will need to count sum of sails for each item. Create a pie chart for each category.

# Group data by category and item, and sum purchase amount
category_item_sales = df.groupby(['Category', 'Item Purchased'])['Purchase Amount (USD)'].count().reset_index(name='Sales Count')

# Find the best-selling item for each category
best_selling_items = category_item_sales.groupby('Category')['Sales Count'].idxmax()

# Create a dictionary to store pie chart figures for each category
category_pie_charts = {}

# Create pie charts for each category
for category in df['Category'].unique():
  category_data = category_item_sales[category_item_sales['Category'] == category]
  fig = px.pie(category_data, values='Sales Count', names='Item Purchased',
               title=f'Best-Selling Items in {category}')
  category_pie_charts[category] = fig

# Show the pie charts
for category, fig in category_pie_charts.items():
  fig.show()


In [9]:
# prompt: Analyze my df and find the best and worst items (item purchased) for each category. You will need to count the average review rating for each item.

# Group data by category and item, and calculate average review rating
category_item_ratings = df.groupby(['Category', 'Item Purchased'])['Review Rating'].mean().reset_index(name='Average Rating')

# Find the best and worst rated item for each category
best_rated_items = category_item_ratings.loc[category_item_ratings.groupby('Category')['Average Rating'].idxmax()]
worst_rated_items = category_item_ratings.loc[category_item_ratings.groupby('Category')['Average Rating'].idxmin()]

# Display the results
print("Best Rated Items:\n", best_rated_items)
print("\nWorst Rated Items:\n", worst_rated_items)


Best Rated Items:
        Category Item Purchased  Average Rating
2   Accessories         Gloves        3.864286
18     Clothing        T-shirt        3.782993
20     Footwear        Sandals        3.841250
24    Outerwear         Jacket        3.763190

Worst Rated Items:
        Category Item Purchased  Average Rating
6   Accessories          Scarf        3.700000
13     Clothing          Shirt        3.629586
21     Footwear          Shoes        3.747333
23    Outerwear           Coat        3.730435


# Third Prompt

from my df analyze previous purchases column  and show me the most loyal customers and their location

In [12]:
# prompt: from my df analyze previous purchases column  and show me the most loyal customers and their location

# Calculate the number of previous purchases for each customer
customer_purchases = df.groupby(['Customer ID', 'Location'])['Previous Purchases'].sum().reset_index()

# Find the most loyal customers (top 20)
most_loyal_customers = customer_purchases.sort_values('Previous Purchases', ascending=False).head(20)

# Display the results
print("Most Loyal Customers:\n", most_loyal_customers)


Most Loyal Customers:
       Customer ID      Location  Previous Purchases
3261         3262  Rhode Island                  50
633           634     Minnesota                  50
2262         2263       Arizona                  50
2264         2265  North Dakota                  50
124           125        Nevada                  50
665           666    New Mexico                  50
3581         3582       Georgia                  50
2099         2100      Oklahoma                  50
313           314       Vermont                  50
3205         3206      Colorado                  50
310           311       Montana                  50
1850         1851       Wyoming                  50
1537         1538        Alaska                  50
722           723       Wyoming                  50
2288         2289    California                  50
2296         2297    Washington                  50
1521         1522    Washington                  50
101           102  North Dakota          